# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is Our team's mission proposal. 

> **Team size:** 3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Leila Rahimiyadkuri||
| Member | Forough Asgari| |
| Member | Sara Davoodabadi | |


## 2. Mission Title & Research Question

**Title:**  
Predicting and Explaining Customer Churn in Retail Banking: Combining Causal Inference, Supervised Classification, and Customer Segmentation

**Research question:**  
Does a customer's account balance or credit score *causally* increase their probability of leaving the bank and if so, does this effect differ across distinct customer segments identified through unsupervised learning?

**Why it matters:**  
Banks typically spend far more to acquire a new customer than to keep an existing one. Most churn studies focus on prediction alone, they tell you *who* is likely to leave but not *why*. We want to go one step further: by combining a causal lens with predictive modeling and customer clustering, we hope to give actionable insight into which financial levers (like improving someone's credit score or offering a better savings rate) could realistically reduce churn. That distinction between correlation and causation matters a lot when a bank is deciding where to spend money on retention.

## 3. Data

**Source:**  
The Bank Customer Churn Dataset, an open-source retail banking dataset hosted on Kaggle.

Link: https://www.kaggle.com/datasets/shantanudhakadd/bank-customer-churn-prediction

**Unit of observation:** One row = one individual retail bank customer. 

The dataset contains 10,000 rows and 14 columns.

**Key variables:**

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| RowNumber | Integer | - | Row index  - carries no information for using in models |
| CustomerId | Integer | - | identifier - carries no information for using in models |
| Surname | String | - | Customer's last name - carries no information for using in models |
| CreditScore | Integer | Treatment(W₁) | Credit score of the customer |
| Geography | Categorical | Feature | Country of residence: France, Germany, or Spain |
| Gender | Categorical | Feature | Customer's gender (Male / Female) |
| Age | Integer | Confounder(X₁) | The age of the customer |
| Tenure | Integer | Feature | Number of years the customer has been with the bank (0–10) |
| Balance | Float | Treatment (W₂) | Current account balance|
| NumOfProducts | Integer | Feature | Number of bank product facilities customer is using |
| HasCrCard | Binary (0/1) | Feature | Whether the customer holds a credit card (1 = yes) |
| IsActiveMember | Binary (0/1) | Feature | Whether the customer has been actively transacting (1 = active) |
| EstimatedSalary | Float | Confounder(X₂) | Annual estimated salary |
| Exited | Binary (0/1) | Target (Y) | Whether the customer closed their account (1 = churned, 0 = stayed) |

**Potential data quality issues:**  
The most notable issue is class imbalance: in most retail churn datasets, churners (Exited = 1) make up only around 20% of records. We will not oversample during training, but we will use class-weighted models and report F1-Score and AUC-ROC rather than raw accuracy to avoid being misled by a majority-class baseline. We will also check for any zero-balance customers, who may represent inactive accounts rather than genuine churn risk, and handle them explicitly in the analysis.

In [10]:
# ── Proposal code block ───────────────────────────────────────────────
# Data loading ────────────────────────────────
import os
import pandas as pd
import numpy as np
import kagglehub

# Download dataset
path = kagglehub.dataset_download("shantanudhakadd/bank-customer-churn-prediction")
print("Path to dataset files:", path)

#Load datset
dataChurn = pd.read_csv(os.path.join(path, 'Churn_Modelling.csv'))

# --- Shape & types
print("\n---------- Dataset Shape ---------")
print(dataChurn.shape)

# --- Data samples
print("\n---------- First and last 5 Rows ----------")
display(dataChurn.head(5))
display(dataChurn.tail(5))

# --- More dtail of Data
print("\n------- Column Types & Missing Values --------")
print(dataChurn.info())

print("\n------------- Summary Statistics -----------")
print(dataChurn.describe().T)

print("\n----------- Churn Rate -------------")
print(dataChurn['Exited'].value_counts(normalize=True).rename({0: 'Stayed', 1: 'Churned'}))



Path to dataset files: C:\Users\Leila\.cache\kagglehub\datasets\shantanudhakadd\bank-customer-churn-prediction\versions\1

---------- Dataset Shape ---------
(10000, 14)

---------- First and last 5 Rows ----------


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1
9999,10000,15628319,Walker,792,France,Female,28,4,130142.79,1,1,0,38190.78,0



------- Column Types & Missing Values --------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB
None

------------- Summary Statistics -------

## 4. Planned Methods

### 4a. Causal Inference

-  Causal graph / DAG (DoWhy)
-  Backdoor adjustment


**Justification:**  
Financial metrics like account balances are heavily confounded by user demographic markers such as age (Age) and baseline earnings (EstimatedSalary). Wealthier or older clients naturally hold higher account balances and may display fundamentally different banking loyalty patterns. We will use the DoWhy package to map out an explicit Directed Acyclic Graph (DAG). By implementing a Backdoor Adjustment, we will formally condition on income and age to capture the true, unconfounded causal impact of financial variables (Balance, CreditScore) on the decision to exit the bank.

We'll construct a DAG using the DoWhy library with the following structure:
- EstimatedSalary → Balance, CreditScore, Exited
- Age → Balance, CreditScore, Exited
- Balance → Exited
- CreditScore → Exited

We'll then apply backdoor adjustment, conditioning on Age and EstimatedSalary, to estimate the average treatment effect (ATE) of Balance and CreditScore on churn. We'll validate this using DoWhy's refutation tests.

---

### 4b. Supervised Learning

- Logistic regression
- Decision Tree / Random Forest


**Justification:**  
Churn prediction is a binary classification problem, so regression isn't the right fit here. We'll start with logistic regression as a transparent baseline, it gives us interpretable log-odds coefficients for each feature, which connects nicely to our causal analysis. Then we'll train a Random Forest, which handles non-linear interactions naturally. We'll use stratified k-fold cross-validation throughout and  deal with the imbalance issue.

---

### 4c. Unsupervised Learning

-  K-Means clustering


**Justification:**  
Customers are not a homogeneous group. An inactive customer with a large balance is a very different churn risk than an active customer who holds multiple products. Before fitting a single global model, we want to understand whether the data naturally groups into distinct customer types.

We'll apply K-Means on a scaled feature set (Balance, CreditScore, NumOfProducts, IsActiveMember, Age) and choose k using the elbow method and silhouette score. The resulting cluster labels will then serve two purposes: first, as an additional feature in our supervised classifiers; second, as a stratification variable in our causal analysis. We want to know whether the causal effect of balance on churn is the same for all customer types, or whether it differs significantly by segment.

## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*



**Predictive models:**  
Given the class imbalance, we'll report AUC-ROC as our primary metric, alongside F1-Score and Recall. Recall matters here because missing a true churner is more costly than a false alarm, a bank would rather send an unnecessary retention offer than lose a customer entirely.

**Causal estimates:**  
We'll run two DoWhy refutation tests:  
1. *Random common cause*  adds a random variable as a fake confounder; our estimated effect should not change significantly.  
2. *Placebo treatment*  replaces the treatment with a random variable; the estimated effect should collapse to near zero.  

If both tests pass, we have reasonable confidence the DAG is correctly specified.

**Clustering:**  
We'll select k using the elbow plot and silhouette coefficient. Cluster quality will also be evaluated descriptively, each cluster should be interpretable in terms of customer characteristics (e.g., "high-balance inactive customers").

**Cross-block connection:**  
The key evaluation question we want to answer at the end is: do the customer segments identified by K-Means moderate the causal effect of balance/credit score on churn? If so, that's a concrete, actionable finding for a bank.


## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | All Members | Data collection & cleaning |
| 2 | Leila | EDA |
| 3 | Leila | Causal inference block |
| 4 | Sara | Supervised learning block |
| 5 | Forough | Unsupervised / generative block |
| 6 | All Members| Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
